In [1]:
# Imports and connection helper

from dotenv import load_dotenv
import os
import snowflake.connector
import pandas as pd

load_dotenv()

def get_connection():
    return snowflake.connector.connect(
        user=os.getenv('SNOWFLAKE_USER'),
        password=os.getenv('SNOWFLAKE_PASSWORD'),
        account=os.getenv('SNOWFLAKE_ACCOUNT'),
        authenticator='username_password_mfa',
        database='DATACO_DB',
        schema='SUPPLY_CHAIN',
        warehouse='DATACO_WH'
    )

print("✅ Helper function ready")

✅ Helper function ready


In [2]:
# Setup Snowflake environment

conn = get_connection()
cursor = conn.cursor()

setup_commands = [
    "USE DATABASE DATACO_DB",
    "USE SCHEMA SUPPLY_CHAIN",
    "USE WAREHOUSE DATACO_WH"
]

for command in setup_commands:
    cursor.execute(command)
    print(f"✅ {command}")

print("\n✅ Snowflake environment ready!")

✅ USE DATABASE DATACO_DB
✅ USE SCHEMA SUPPLY_CHAIN
✅ USE WAREHOUSE DATACO_WH

✅ Snowflake environment ready!


In [32]:
# Create tables

tables = {

    "dim_customer": """
        CREATE TABLE IF NOT EXISTS dim_customer (
            customer_id         INTEGER PRIMARY KEY,
            first_name          VARCHAR,
            last_name           VARCHAR,
            customer_email      VARCHAR,
            customer_segment    VARCHAR,
            customer_city       VARCHAR,
            customer_state      VARCHAR,
            customer_country    VARCHAR,
            customer_street     VARCHAR,
            customer_zipcode    VARCHAR
        )
    """,

    "dim_category": """
        CREATE TABLE IF NOT EXISTS dim_category (
            category_id         INTEGER PRIMARY KEY,
            category_name       VARCHAR NOT NULL
        )
    """,

    "dim_department": """
        CREATE TABLE IF NOT EXISTS dim_department (
            department_id       INTEGER PRIMARY KEY,
            department_name     VARCHAR NOT NULL
        )
    """,

    "dim_product": """
        CREATE TABLE IF NOT EXISTS dim_product (
            product_card_id     INTEGER PRIMARY KEY,
            product_name        VARCHAR,
            product_image       VARCHAR,
            product_price       FLOAT,
            product_status      INTEGER,
            category_id         INTEGER
        )
    """,

    "dim_location": """
        CREATE TABLE IF NOT EXISTS dim_location (
            location_id         INTEGER AUTOINCREMENT PRIMARY KEY,
            market              VARCHAR,
            order_region        VARCHAR,
            order_country       VARCHAR,
            order_state         VARCHAR,
            order_city          VARCHAR,
            latitude            FLOAT,
            longitude           FLOAT
        )
    """,

    "fact_orders": """
        CREATE TABLE IF NOT EXISTS fact_orders (
            order_item_id               INTEGER PRIMARY KEY,
            order_id                    INTEGER,
            order_customer_id           INTEGER,
            customer_id                 INTEGER,
            product_card_id             INTEGER,
            department_id               INTEGER,
            location_id                 INTEGER,
            shipping_mode               VARCHAR,
            date_id                     INTEGER,
            order_date                  DATE,
            shipping_date               DATE,
            real_shipping_days          INTEGER,
            scheduled_shipment_days     INTEGER,
            delay_days                  INTEGER,
            late_delivery_risk          INTEGER,
            is_late                     INTEGER,
            delivery_status             VARCHAR,
            order_status                VARCHAR,
            transaction_type            VARCHAR,
            order_item_quantity         INTEGER,
            order_item_discount         FLOAT,
            order_item_discount_rate    FLOAT,
            order_item_product_price    FLOAT,
            order_item_profit_ratio     FLOAT,
            sales                       FLOAT,
            order_item_total            FLOAT,
            order_profit_per_order      FLOAT,
            benefit_per_order           FLOAT,
            sales_per_customer          FLOAT
        )
    """
}

for table_name, ddl in tables.items():
    cursor.execute(ddl)
    print(f"✅ Created: {table_name}")

print("\n✅ All tables created!")

✅ Created: dim_customer
✅ Created: dim_category
✅ Created: dim_department
✅ Created: dim_product
✅ Created: dim_location
✅ Created: fact_orders

✅ All tables created!


In [6]:
data_df = pd.read_pickle('./data/dataco_cleaned.pkl')

data_df.shape

(180519, 49)

In [8]:
data_df.head()

,Type,Days for shipping (real),Days for shipment (scheduled),Benefit per order,Sales per customer,Delivery Status,Late_delivery_risk,Category Id,Category Name,Customer City,...,Order Status,Product Card Id,Product Image,Product Name,Product Price,Product Status,shipping date (DateOrders),Shipping Mode,delay_days,is_late
0,DEBIT,3,4,91.250000,314.640015,Advance shipping,0,73,Sporting Goods,Caguas,...,COMPLETE,1360,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,2018-02-03 22:56:00,Standard Class,-1,0
1,TRANSFER,5,4,-249.089996,311.359985,Late delivery,1,73,Sporting Goods,Caguas,...,PENDING,1360,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,2018-01-18 12:27:00,Standard Class,1,1
2,CASH,4,4,-247.779999,309.720001,Shipping on time,0,73,Sporting Goods,San Jose,...,CLOSED,1360,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,2018-01-17 12:06:00,Standard Class,0,0
3,DEBIT,3,4,22.860001,304.809998,Advance shipping,0,73,Sporting Goods,Los Angeles,...,COMPLETE,1360,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,2018-01-16 11:45:00,Standard Class,-1,0
4,PAYMENT,2,4,134.210007,298.250000,Advance shipping,0,73,Sporting Goods,Caguas,...,PENDING_PAYMENT,1360,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,2018-01-15 11:24:00,Standard Class,-2,0


## Creating Tables

In [13]:
# dim_customer

dim_customer_cols = {
    'Customer Id':      'customer_id',
    'Customer Fname':   'first_name',
    'Customer Lname':   'last_name',
    'Customer Segment': 'customer_segment',
    'Customer City':    'customer_city',
    'Customer State':   'customer_state',
    'Customer Country': 'customer_country',
    'Customer Street':  'customer_street',
    'Customer Zipcode': 'customer_zipcode'
}

dim_customer = (
    data_df[list(dim_customer_cols.keys())]
    .drop_duplicates(subset=['Customer Id'])
    .rename(columns=dim_customer_cols)
    .reset_index(drop=True)
)

dim_customer

,customer_id,first_name,last_name,customer_segment,customer_city,customer_state,customer_country,customer_street,customer_zipcode
0,20755,Cally,Holloway,Consumer,Caguas,PR,Puerto Rico,5365 Noble Nectar Island,725.0
1,19492,Irene,Luna,Consumer,Caguas,PR,Puerto Rico,2679 Rustic Loop,725.0
2,19491,Gillian,Maldonado,Consumer,San Jose,CA,EE. UU.,8510 Round Bear Gate,95125.0
3,19490,Tana,Tate,Home Office,Los Angeles,CA,EE. UU.,3200 Amber Bend,90027.0
4,19489,Orli,Hendricks,Corporate,Caguas,PR,Puerto Rico,8671 Iron Anchor Corners,725.0
...,...,...,...,...,...,...,...,...,...
20647,18586,Diana,Tran,Home Office,Caguas,PR,Puerto Rico,1802 Broad Carrefour,725.0
20648,16322,Anika,Davenport,Home Office,Caguas,PR,Puerto Rico,8284 Colonial Gate Path,725.0
20649,18579,Yuri,Smith,Home Office,Caguas,PR,Puerto Rico,2248 Heather Mews,725.0
20650,16244,Hyacinth,Witt,Home Office,Caguas,PR,Puerto Rico,7247 Honey Prairie Field,725.0


In [11]:
# dim_category

dim_category_cols = {
    'Category Id':   'category_id',
    'Category Name': 'category_name'
}

dim_category = (
    data_df[list(dim_category_cols.keys())]
    .drop_duplicates(subset=['Category Id'])
    .rename(columns=dim_category_cols)
    .reset_index(drop=True)
)

dim_category

,category_id,category_name
0,73,Sporting Goods
1,17,Cleats
2,29,Shop By Sport
3,24,Women's Apparel
4,13,Electronics
5,12,Boxing & MMA
6,9,Cardio Equipment
7,41,Trade-In
8,37,Electronics
9,38,Kids' Golf Clubs


In [17]:
# dim_department

dim_department_cols = {
    'Department Id':   'department_id',
    'Department Name': 'department_name'
}

dim_department = (
    data_df[list(dim_department_cols.keys())]
    .drop_duplicates(subset=['Department Id'])
    .rename(columns=dim_department_cols)
    .reset_index(drop=True)
)

dim_department

,department_id,department_name
0,2,Fitness
1,4,Apparel
2,5,Golf
3,3,Footwear
4,6,Outdoors
5,7,Fan Shop
6,10,Technology
7,8,Book Shop
8,9,Discs Shop
9,11,Pet Shop


In [18]:
# dim_product

dim_product_cols = {
    'Product Card Id': 'product_card_id',
    'Product Name':    'product_name',
    'Product Image':   'product_image',
    'Product Price':   'product_price',
    'Product Status':  'product_status',
    'Category Id':     'category_id'
}

dim_product = (
    data_df[list(dim_product_cols.keys())]
    .drop_duplicates(subset=['Product Card Id'])
    .rename(columns=dim_product_cols)
    .reset_index(drop=True)
)

dim_product

,product_card_id,product_name,product_image,product_price,product_status,category_id
0,1360,Smart watch,http://images.acmesports.sports/Smart+watch,327.750000,0,73
1,365,Perfect Fitness Perfect Rip Deck,http://images.acmesports.sports/Perfect+Fitnes...,59.990002,0,17
2,627,Under Armour Girls' Toddler Spine Surge Runni,http://images.acmesports.sports/Under+Armour+G...,39.990002,0,29
3,502,Nike Men's Dri-FIT Victory Golf Polo,http://images.acmesports.sports/Nike+Men%27s+D...,50.000000,0,24
4,278,Under Armour Men's Compression EV SL Slide,http://images.acmesports.sports/Under+Armour+M...,44.990002,0,13
...,...,...,...,...,...,...
113,646,Merrell Women's Grassbow Sport Hiking Shoe,http://images.acmesports.sports/Merrell+Women%...,99.989998,0,30
114,1361,Toys,http://images.acmesports.sports/Toys,11.540000,0,74
115,1073,Pelican Sunstream 100 Kayak,http://images.acmesports.sports/Pelican+Sunstr...,199.990005,0,48
116,1059,Pelican Maverick 100X Kayak,http://images.acmesports.sports/Pelican+Maveri...,349.989990,0,48


In [28]:
# dim_location

dim_location_cols = {
    'Market':        'market',
    'Order Region':  'order_region',
    'Order Country': 'order_country',
    'Order State':   'order_state',
    'Order City':    'order_city',
    'Latitude':      'latitude',
    'Longitude':     'longitude'
}

dim_location = (
    data_df[list(dim_location_cols.keys())]
    .drop_duplicates()
    .rename(columns=dim_location_cols)
    .reset_index(drop=True)
)

dim_location

,market,order_region,order_country,order_state,order_city,latitude,longitude
0,Pacific Asia,Southeast Asia,Indonesia,Java Occidental,Bekasi,18.251453,-66.037056
1,Pacific Asia,South Asia,India,Rajastán,Bikaner,18.279451,-66.037064
2,Pacific Asia,South Asia,India,Rajastán,Bikaner,37.292233,-121.881279
3,Pacific Asia,Oceania,Australia,Queensland,Townsville,34.125946,-118.291016
4,Pacific Asia,Oceania,Australia,Queensland,Townsville,18.253769,-66.037048
...,...,...,...,...,...,...,...
64862,Pacific Asia,Southeast Asia,Vietnam,Ciudad Ho Chi Minh,Ho Chi Minh City,18.209421,-66.370560
64863,Pacific Asia,Eastern Asia,China,Hubei,Ezhou,39.935619,-83.138512
64864,Pacific Asia,Southeast Asia,Malasia,Sabah,Tawau,33.710079,-84.112473
64865,Pacific Asia,Southeast Asia,Indonesia,Java Central,Semarang,18.227577,-66.043983


In [33]:
fact_orders_cols = {
    'Order Item Id':                 'order_item_id',
    'Order Id':                      'order_id',
    'Order Customer Id':             'order_customer_id',
    'Customer Id':                   'customer_id',
    'Product Card Id':               'product_card_id',
    'Department Id':                 'department_id',
    'Market':                        'market',
    'Order Region':                  'order_region',
    'Order Country':                 'order_country',
    'Order State':                   'order_state',
    'Order City':                    'order_city',
    'Shipping Mode':                 'shipping_mode',
    'order date (DateOrders)':       'order_date',
    'shipping date (DateOrders)':    'shipping_date',
    'Days for shipping (real)':      'real_shipping_days',
    'Days for shipment (scheduled)': 'scheduled_shipment_days',
    'delay_days':                    'delay_days',
    'Late_delivery_risk':            'late_delivery_risk',
    'is_late':                       'is_late',
    'Delivery Status':               'delivery_status',
    'Order Status':                  'order_status',
    'Type':                          'transaction_type',
    'Order Item Quantity':           'order_item_quantity',
    'Order Item Discount':           'order_item_discount',
    'Order Item Discount Rate':      'order_item_discount_rate',
    'Order Item Product Price':      'order_item_product_price',
    'Order Item Profit Ratio':       'order_item_profit_ratio',
    'Sales':                         'sales',
    'Order Item Total':              'order_item_total',
    'Order Profit Per Order':        'order_profit_per_order',
    'Benefit per order':             'benefit_per_order',
    'Sales per customer':            'sales_per_customer'
}

fact_orders = (
    data_df[list(fact_orders_cols.keys())]
    .rename(columns=fact_orders_cols)
    .reset_index(drop=True)
)

fact_orders['order_date'] = pd.to_datetime(fact_orders['order_date']).dt.date
fact_orders['shipping_date'] = pd.to_datetime(fact_orders['shipping_date']).dt.date

fact_orders

,order_item_id,order_id,order_customer_id,customer_id,product_card_id,department_id,market,order_region,order_country,order_state,...,order_item_quantity,order_item_discount,order_item_discount_rate,order_item_product_price,order_item_profit_ratio,sales,order_item_total,order_profit_per_order,benefit_per_order,sales_per_customer
0,180517,77202,20755,20755,1360,2,Pacific Asia,Southeast Asia,Indonesia,Java Occidental,...,1,13.110000,0.04,327.750000,0.29,327.750000,314.640015,91.250000,91.250000,314.640015
1,179254,75939,19492,19492,1360,2,Pacific Asia,South Asia,India,Rajastán,...,1,16.389999,0.05,327.750000,-0.80,327.750000,311.359985,-249.089996,-249.089996,311.359985
2,179253,75938,19491,19491,1360,2,Pacific Asia,South Asia,India,Rajastán,...,1,18.030001,0.06,327.750000,-0.80,327.750000,309.720001,-247.779999,-247.779999,309.720001
3,179252,75937,19490,19490,1360,2,Pacific Asia,Oceania,Australia,Queensland,...,1,22.940001,0.07,327.750000,0.08,327.750000,304.809998,22.860001,22.860001,304.809998
4,179251,75936,19489,19489,1360,2,Pacific Asia,Oceania,Australia,Queensland,...,1,29.500000,0.09,327.750000,0.45,327.750000,298.250000,134.210007,134.210007,298.250000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
180514,65177,26043,1005,1005,1004,7,Pacific Asia,Eastern Asia,China,Shanghái,...,1,0.000000,0.00,399.980011,0.10,399.980011,399.980011,40.000000,40.000000,399.980011
180515,65161,26037,9141,9141,1004,7,Pacific Asia,Eastern Asia,Japón,Osaka,...,1,4.000000,0.01,399.980011,-1.55,399.980011,395.980011,-613.770019,-613.770019,395.980011
180516,65129,26024,291,291,1004,7,Pacific Asia,Oceania,Australia,Australia del Sur,...,1,8.000000,0.02,399.980011,0.36,399.980011,391.980011,141.110001,141.110001,391.980011
180517,65126,26022,2813,2813,1004,7,Pacific Asia,Oceania,Australia,Australia del Sur,...,1,12.000000,0.03,399.980011,0.48,399.980011,387.980011,186.229996,186.229996,387.980011


In [38]:
from snowflake.connector.pandas_tools import write_pandas

conn = get_connection()

tables_to_load = {
    'DIM_CUSTOMER':   dim_customer,
    'DIM_CATEGORY':   dim_category,
    'DIM_DEPARTMENT': dim_department,
    'DIM_PRODUCT':    dim_product,
    'DIM_LOCATION':   dim_location,
    'FACT_ORDERS':    fact_orders
}

for table_name, df in tables_to_load.items():
    
    # Uppercase column names to match Snowflake
    df_upload = df.copy()
    df_upload.columns = df_upload.columns.str.upper()
    
    success, chunks, rows, output = write_pandas(
        conn=conn,
        df=df_upload,
        table_name=table_name,
        database='DATACO_DB',
        schema='SUPPLY_CHAIN',
        auto_create_table=True,
        overwrite=True,
        quote_identifiers=False
    )
    print(f"✅ {table_name}: {rows:,} rows — success: {success}")

conn.close()
print("\n✅ All tables loaded!")

✅ DIM_CUSTOMER: 20,652 rows — success: True
✅ DIM_CATEGORY: 51 rows — success: True
✅ DIM_DEPARTMENT: 11 rows — success: True
✅ DIM_PRODUCT: 118 rows — success: True
✅ DIM_LOCATION: 64,867 rows — success: True
✅ FACT_ORDERS: 180,519 rows — success: True

✅ All tables loaded!
